# 04 - Inference Demo (MC Dropout)

**Purpose:** Simulate online degradation monitoring for one test engine with uncertainty-aware RUL estimates.

**Inputs:** `model/best_model.pth` and `data/processed/test_engine_trajectories.pkl`

**Outputs:** time-series plot of predicted RUL with 95% confidence band and final-step prediction summary.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from model.lstm_model import LSTMRULModel, mc_dropout_predict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "model" / "best_model.pth"
DEVICE = torch.device("cpu")

In [ ]:
with open(PROCESSED_DIR / "test_engine_trajectories.pkl", "rb") as fp:
    test_trajectories = pickle.load(fp)

engine_ids = sorted(test_trajectories.keys())
candidate = engine_ids[len(engine_ids) // 2]  # representative mid-range engine
payload = test_trajectories[candidate]

model = LSTMRULModel(input_size=14, hidden_size=64, num_layers=2, dropout=0.3).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

sequences = torch.tensor(payload["sequences"], dtype=torch.float32, device=DEVICE)
cycles = np.array(payload["cycles"])
actual_rul = np.array(payload["actual_rul"])

mean_rul, std_rul = mc_dropout_predict(model, sequences, n_passes=50)
mean_rul = mean_rul.numpy()
std_rul = std_rul.numpy()

lower = mean_rul - 1.96 * std_rul
upper = mean_rul + 1.96 * std_rul

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(cycles, actual_rul, label="Actual RUL", color="#2563eb", linewidth=1.7)
plt.plot(cycles, mean_rul, label="Predicted mean RUL", color="#f59e0b", linewidth=1.7)
plt.fill_between(cycles, lower, upper, color="#f59e0b", alpha=0.25, label="95% CI")
plt.title(f"Engine {candidate}: MC Dropout Inference Over Time")
plt.xlabel("Cycle")
plt.ylabel("RUL")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
final_mean = float(mean_rul[-1])
final_std = float(std_rul[-1])
final_ci = (final_mean - 1.96 * final_std, final_mean + 1.96 * final_std)

print(f"Engine ID: {candidate}")
print(f"Final timestep predicted RUL: {final_mean:.2f}")
print(f"Final timestep uncertainty (std): {final_std:.2f}")
print(f"Final timestep 95% CI: [{final_ci[0]:.2f}, {final_ci[1]:.2f}]")